# Milestone 2 — Phân tích LaTeX, chuẩn hóa và khử trùng

**CSC14119 — Nhập môn Khoa học Dữ liệu · Nhóm 3**

[![Mở trên Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DuyNhat1705/Intro2DS-Group3-Source/blob/main/src/parser.ipynb)

Notebook chạy pipeline trong `parsing.md`: **gom file → làm sạch → dựng cây và trích tài liệu tham khảo → hợp nhất bibliography → đổi khóa citation → khử trùng element → xuất bảng và thống kê**.

Đầu vào là corpus Milestone 1 đã có; notebook không tải lại arXiv. Chỉnh ô cấu hình rồi chọn **Run all**. Mặc định chạy thử 20 ID đầu của R1; đặt `SAMPLE_N = None` để chạy toàn bộ phạm vi. SaT chạy CPU và tải trọng số ở lần tách câu đầu tiên nếu máy chưa có cache.

In [ ]:
# CẤU HÌNH — chỉnh trước khi Run all
from pathlib import Path

RANGES = {
    1: ("2609", 27834, 32833),
    2: ("2609", 17834, 22833),
    3: ("2608", 14007, 19006),
    4: ("2608", 4007, 9006),
    5: ("2511", 6709, 11708),
}
RUN_RANGES = "local"          # [3], [1, 2], "all" hoặc "local" (tìm bài từ thư mục/state)
SAMPLE_N = None             # số ID đầu mỗi range; None = toàn bộ; local: giới hạn tổng số bài
WORKER = None             # None: tự đặt theo range; dùng tên khác nhau nếu chạy đồng thời

REPO_DIR = None           # local: tự tìm repo từ thư mục hiện tại; hoặc đường dẫn tuyệt đối
REPO_URL = "https://github.com/DuyNhat1705/Intro2DS-Group3-Source.git"
BRANCH = "Testing"        # nhánh Colab clone; "main" sau khi đã kiểm thử xong
INSTALL_DEPENDENCIES = None  # None: tự cài trên Colab, chỉ kiểm tra trên local
MOUNT_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/Intro2DS-Group3"
DATA_DIR_OVERRIDE = None     # ví dụ r"C:/Intro2DS-Group3-Source/data"
DATA_TAR = None              # None: trên Colab tự dùng data.tar.gz/.tgz/.tar trong DRIVE_PROJECT_DIR nếu có;
                             # đường dẫn cụ thể (tương đối tính từ DRIVE_PROJECT_DIR); False = luôn đọc thư mục data
LOCAL_DATA_DIR = "/content/parser_data"  # nơi giải nén bản tar; ổ local, mất khi runtime bị reset
OUTPUT_DIR_OVERRIDE = None   # mặc định statistics_data/parser, tách khỏi output scraper
SOURCE_STATE_PATHS = None    # None: tự đọc data/source_state.json và data/.state/**/source_state.json

RUN_SMOKE_TEST = True
RUN_PIPELINE = True
VERIFY_CHECKPOINTS = "size"  # "size": so kích thước tệp output (nhanh trên Drive); "full": băm lại toàn bộ,
                             # phát hiện cả tệp bị sửa mà không đổi kích thước — nên dùng cho lần chạy cuối
RESUME = True               # chỉ dùng lại checkpoint ok/empty khi nguồn, mã và tùy chọn không đổi
ENABLE_ENRICHMENT = False   # mặc định offline; True cho phép CrossRef và LLM nếu đã có key
FUZZY_THRESHOLD = 0.92
REFINE_BIBTEX = True        # hậu xử lý bibtex sau parse (mục 4b); có cache nên chạy lại rất nhanh
REFINE_WITH_ANYSTYLE = True # tách lại \bibitem đáng ngờ; Colab tự cài Ruby + anystyle-cli (vài phút)
REFINE_CACHE_DIR = None     # None: <OUTPUT_DIR>/refine_cache; giữ trên Drive để dùng lại giữa các phiên
NUM_WORKERS = "auto"        # tự chọn theo CPU/RAM; 1 = tuần tự; số nguyên = giới hạn mong muốn
MAX_WORKERS = None          # giới hạn thêm số tiến trình; None = dùng giới hạn CPU
ADAPTIVE_RAM = True         # tự điều chỉnh lượng công việc và thu hồi worker rảnh khi thiếu RAM
RESERVE_RAM_GB = None       # None: tự chừa RAM cho hệ thống; hoặc số GiB muốn chừa
RAM_PER_WORKER_GB = 1.5     # mức dự trù tối thiểu mỗi worker, sẽ tăng theo RSS đo được
RAM_WAIT_TIMEOUT_SEC = 60   # hết thời gian chờ RAM thì lưu phần đã xong, để phần còn lại chạy tiếp sau
PAPER_TIMEOUT_SEC = 900     # một bài chạy quá số giây này thì bị dừng và ghi lỗi; None = không giới hạn (chỉ khi song song)
MAX_RUN_HOURS = 11.0        # kiểm tra giữa các bài; None = không giới hạn
MONITOR_EVERY_SEC = 1.0
DISK_EVERY_PAPERS = 500
REPORT_ALL_WORKERS = False  # ghép số liệu các worker sau khi đã gom output về cùng thư mục

## 1. Cách chạy trên local và Google Colab

**Local:** dùng Python 3.12 trở lên, cài `pip install -r src/requirements.txt`, mở notebook từ repo hoặc đặt `REPO_DIR`. Dữ liệu mặc định ở `<repo>/data`. Muốn xử lý thư mục tự chọn, đặt `DATA_DIR_OVERRIDE` và `RUN_RANGES = "local"`.

**Colab:** chọn runtime CPU. Notebook clone repo nếu chưa có, cài các thư viện parser theo phiên bản của `src/requirements.txt`, rồi mount Drive. Cần đẩy mã parser lên nhánh `BRANCH` trước khi mở ở Colab. Nếu repo đã tồn tại, notebook dùng mã hiện có và không tự pull đè thay đổi; khi bản clone đó đang ở nhánh khác `BRANCH`, notebook dừng và yêu cầu xóa thư mục repo hoặc checkout đúng nhánh, để không chạy nhầm mã của nhánh khác. Mặc định đọc `MyDrive/Intro2DS-Group3/data` và ghi `MyDrive/Intro2DS-Group3/statistics_data/parser`.

**Nguồn dữ liệu dạng nén (tùy chọn):** nếu `DRIVE_PROJECT_DIR` chứa `data.tar.gz`, `data.tgz` hoặc `data.tar`, notebook giải nén tệp này vào `LOCAL_DATA_DIR` trên ổ của runtime và đọc dữ liệu từ đó; nếu không có, notebook đọc trực tiếp thư mục `data` trên Drive. Bản giải nén được dùng lại trong cùng runtime khi tệp nén không đổi. Output luôn ghi lên Drive để checkpoint còn lại sau khi runtime bị reset. Tệp nén là ảnh chụp của `data` tại thời điểm tạo, nên cần tạo lại khi dữ liệu nguồn thay đổi. Chữ ký checkpoint không phụ thuộc vị trí thư mục dữ liệu, nên chuyển giữa hai nguồn không làm mất checkpoint.

**Kiểm thử:** `python -m pytest tests -q` chạy bộ kiểm thử của parser (tìm bài, chữ ký checkpoint, khử trùng, hợp nhất reference, chạy tuần tự/song song và giới hạn thời gian). Các kiểm thử giả lập mô hình tách câu nên không cần tải trọng số SaT.

**Sau khi cập nhật mã:** restart kernel/runtime rồi Run all để nạp đúng module mới. Không cần xóa kết quả cũ; chữ ký mã thay đổi sẽ tự làm mất hiệu lực checkpoint cũ. Chỉ bấm Run all trong kernel đang giữ module cũ sẽ không tự nạp lại mã Python.

**Chạy tiếp:** mỗi bài hoàn thành có phân mảnh Parquet và checkpoint riêng. Khi chạy lại, nguồn, cấu hình và tính toàn vẹn các tệp đầu ra được kiểm tra bằng SHA-256; bài `partial/error` được thử lại. Không chạy hai notebook có cùng `WORKER` vào một thư mục output. Có thể dừng bằng Interrupt; bài đang làm chưa hoàn thành sẽ được chạy lại ở phiên sau. Giới hạn giờ chỉ được kiểm tra giữa các bài.

**Song song:** mặc định `NUM_WORKERS="auto"`. Mỗi tiến trình xử lý trọn một bài và tái sử dụng mô hình SaT riêng. Bộ điều phối chạy một bài để đo RAM trước khi tăng số worker, giới hạn theo CPU và RAM khả dụng. Có thể đặt `MAX_WORKERS` để giới hạn thêm, hoặc `NUM_WORKERS=1` để trở lại chế độ tuần tự. Trên Windows/Colab, worker dùng cơ chế spawn từ module Python, không chạy lại các ô notebook.

**Mạng:** `ENABLE_ENRICHMENT=False` tắt enrichment; lần đầu tải trọng số SaT và cài thư viện vẫn cần mạng. Để chạy hoàn toàn offline, cài sẵn thư viện và cache trọng số trước. API key Gemini, nếu dùng, lấy từ biến môi trường hoặc tự nạp vào môi trường từ Colab Secrets; notebook không ghi key vào cấu hình/log.

In [ ]:
import importlib.metadata
import subprocess
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if REPO_DIR is not None:
    REPO_ROOT = Path(REPO_DIR).expanduser().resolve()
elif IN_COLAB:
    REPO_ROOT = Path("/content/Intro2DS-Group3-Source")
    if not REPO_ROOT.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
    else:
        current_branch = subprocess.run(["git", "-C", str(REPO_ROOT), "rev-parse", "--abbrev-ref", "HEAD"],
                                        capture_output=True, text=True).stdout.strip()
        if current_branch != BRANCH:
            raise RuntimeError(f"{REPO_ROOT} đang ở nhánh {current_branch!r}, khác BRANCH={BRANCH!r}. "
                               f"Chạy !rm -rf {REPO_ROOT} (hoặc git checkout {BRANCH}), restart runtime rồi Run all.")
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next((p for p in candidates if (p / "src" / "hierarchy.py").is_file()), None)
    if REPO_ROOT is None:
        raise FileNotFoundError("Không tìm thấy repo. Hãy đặt REPO_DIR ở ô cấu hình.")

required_files = ["parser_pipeline.py", "parser_reporting.py", "parser_parallel.py", "gather_clean_tex.py", "bibtex.py", "hierarchy.py"]
missing_files = [name for name in required_files if not (REPO_ROOT / "src" / name).is_file()]
if missing_files:
    raise FileNotFoundError(f"Nhánh đang dùng thiếu module {missing_files}; cập nhật mã parser trước khi chạy.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

parser_packages = {"pandas", "pyarrow", "matplotlib", "bibtexparser", "psutil", "wtpsplit", "onnxruntime"}
requirements = [line.strip() for line in (REPO_ROOT / "src" / "requirements.txt").read_text().splitlines()
                if line.strip().split("==")[0] in parser_packages]
if {r.split("==")[0] for r in requirements} != parser_packages:
    raise ValueError("requirements.txt chưa định nghĩa đủ các thư viện parser.")
install = IN_COLAB if INSTALL_DEPENDENCIES is None else INSTALL_DEPENDENCIES
versions_before = {name: getattr(sys.modules.get(name), "__version__", None) for name in parser_packages}
if install:
    subprocess.run([sys.executable, "-m", "pip", "install", *requirements], check=True)

dependency_rows = []
for requirement in requirements:
    name, expected = requirement.split("==")
    try:
        actual = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        raise RuntimeError(f"Thiếu {name}. Chạy pip install -r src/requirements.txt rồi khởi động lại kernel.")
    dependency_rows.append({"thư viện": name, "yêu cầu": expected, "đã cài": actual})
    if versions_before[name] is not None and versions_before[name] != actual:
        raise RuntimeError(f"{name} đã được nạp trước khi cập nhật. Restart kernel/runtime rồi Run all lại.")
    if actual != expected:
        print(f"Lưu ý: {name}={actual}, khác bản tái lập {expected}.")
print("Môi trường:", "Google Colab CPU" if IN_COLAB else "Local", "| Repo:", REPO_ROOT)

In [ ]:
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

import shutil
import time

project_root = Path(DRIVE_PROJECT_DIR) if IN_COLAB and MOUNT_DRIVE else REPO_ROOT


# Ghi vào thông tin phiên của run_parser để báo cáo thời gian end-to-end gồm cả bước chuẩn bị dữ liệu.
DATA_PREPARATION = {"data_source": "directory"}


def extracted_data_dir():
    """Giải nén bản tar của data vào ổ local; trả None nếu không dùng tar."""
    if DATA_DIR_OVERRIDE or DATA_TAR is False:
        return None
    if DATA_TAR is None:
        if not (IN_COLAB and MOUNT_DRIVE):
            return None
        found = [project_root / name for name in ("data.tar.gz", "data.tgz", "data.tar") if (project_root / name).is_file()]
        if not found:
            return None
        archive = found[0]
    else:
        archive = Path(DATA_TAR).expanduser()
        archive = archive if archive.is_absolute() else project_root / archive
        if not archive.is_file():
            print(f"Không thấy {archive}; đọc thư mục data như bình thường.")
            return None
    target = Path(LOCAL_DATA_DIR)
    info = archive.stat()
    stamp = f"{archive}|{info.st_size}|{info.st_mtime_ns}"
    marker = target.with_name(target.name + ".source")
    seconds_file = target.with_name(target.name + ".seconds")
    DATA_PREPARATION.update(data_source="tar", archive=archive.name, archive_bytes=info.st_size)
    if target.is_dir() and marker.is_file() and marker.read_text(encoding="utf-8") == stamp:
        print(f"Dùng lại dữ liệu đã giải nén từ {archive.name} tại {target}")
        first = seconds_file.read_text(encoding="utf-8").strip() if seconds_file.is_file() else ""
        DATA_PREPARATION.update(extract_sec_this_session=0.0, reused_extraction=True,
                                extract_sec_first=float(first) if first else None)
    else:
        shutil.rmtree(target, ignore_errors=True)
        marker.unlink(missing_ok=True)
        target.mkdir(parents=True)
        if shutil.disk_usage(target).free < info.st_size:
            print(f"Ổ local không đủ chỗ cho {archive.name}; đọc thư mục data như bình thường.")
            DATA_PREPARATION.clear(); DATA_PREPARATION["data_source"] = "directory"
            return None
        print(f"Giải nén {archive} ({info.st_size / 2**30:.2f} GiB) → {target} ...")
        started = time.perf_counter()
        try:
            subprocess.run(["tar", "-xf", str(archive), "-C", str(target)], check=True)
        except (OSError, subprocess.CalledProcessError) as exc:
            shutil.rmtree(target, ignore_errors=True)
            print(f"Giải nén thất bại ({exc}); đọc thư mục data như bình thường.")
            DATA_PREPARATION.clear(); DATA_PREPARATION["data_source"] = "directory"
            return None
        elapsed = time.perf_counter() - started
        marker.write_text(stamp, encoding="utf-8")
        seconds_file.write_text(f"{elapsed:.3f}", encoding="utf-8")
        DATA_PREPARATION.update(extract_sec_this_session=round(elapsed, 3), reused_extraction=False,
                                extract_sec_first=round(elapsed, 3))
        print(f"Giải nén xong sau {elapsed:.0f}s")
    # Bản tar có thể chứa thư mục data/ ở gốc hoặc chứa thẳng các thư mục bài.
    return target / "data" if (target / "data").is_dir() else target


DATA_DIR = (Path(DATA_DIR_OVERRIDE).expanduser().resolve() if DATA_DIR_OVERRIDE
            else extracted_data_dir() or project_root / "data")
OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE).expanduser().resolve() if OUTPUT_DIR_OVERRIDE else project_root / "statistics_data" / "parser"
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Không có thư mục dữ liệu {DATA_DIR}. Kiểm tra Drive hoặc DATA_DIR_OVERRIDE.")
if OUTPUT_DIR.resolve() == DATA_DIR.resolve():
    raise ValueError("OUTPUT_DIR phải khác DATA_DIR để tách output parser khỏi nguồn scraper.")

import json
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from src.parser_pipeline import discover_publications, parse_publication
from src.parser_reporting import run_parser

display(pd.DataFrame(dependency_rows))
print("Nguồn:", DATA_DIR)
print("Đầu ra:", OUTPUT_DIR)

## 2. Chọn phạm vi và kiểm tra đầu vào

Range giữ nguyên định nghĩa trong `scraper.ipynb`. Mẫu thử chỉ thay đổi phạm vi báo cáo, không được suy diễn thành toàn bộ 25.000 ID. Chế độ `local` tìm bài qua thư mục và state đã có, kể cả bản rỗng chỉ còn state; chế độ range giữ cả ID chưa có dữ liệu để mẫu số độ phủ không bị thu nhỏ.

Danh sách phiên bản lấy từ metadata, state Milestone 1 và thư mục thực tế. Phiên bản đã xác nhận `no_tex` được ghi `empty`; thiếu thư mục không có bằng chứng không được tự coi là rỗng. ID chưa có bất kỳ thông tin phiên bản dùng bản ghi v1 để biểu diễn thiếu đầu vào, không phải bằng chứng bài có một phiên bản. Lỗi metadata, nguồn thiếu và bibliography mơ hồ được giữ trong kết quả kiểm tra/trạng thái.

In [ ]:
if SAMPLE_N is not None and (type(SAMPLE_N) is not int or SAMPLE_N <= 0):
    raise ValueError("SAMPLE_N phải là số nguyên dương hoặc None.")
if RUN_RANGES == "local":
    selected_ranges = []
    # limit giới hạn số bài trước khi đọc metadata và liệt kê phiên bản
    jobs = discover_publications(DATA_DIR, source_state_paths=SOURCE_STATE_PATHS, limit=SAMPLE_N)
else:
    selected_ranges = sorted(RANGES) if RUN_RANGES == "all" else list(RUN_RANGES)
    if not selected_ranges or len(set(selected_ranges)) != len(selected_ranges) or any(r not in RANGES for r in selected_ranges):
        raise ValueError("RUN_RANGES phải là danh sách range hợp lệ, 'all' hoặc 'local'.")
    targets = []
    for r in selected_ranges:
        month, first, last = RANGES[r]
        end = last + 1 if SAMPLE_N is None else min(last + 1, first + SAMPLE_N)
        targets.extend(f"{month}.{number:05d}" for number in range(first, end))
    if len(targets) != len(set(targets)):
        raise ValueError("Định nghĩa các range đang chồng lấn.")
    jobs = discover_publications(DATA_DIR, target_ids=targets, source_state_paths=SOURCE_STATE_PATHS)

def range_label(arxiv_id):
    for r, (month, first, last) in RANGES.items():
        if arxiv_id.startswith(month + ".") and arxiv_id.split(".")[-1].isdigit() and first <= int(arxiv_id.split(".")[-1]) <= last:
            return f"R{r}"
    return "local"

for job in jobs:
    job["range"] = range_label(job["arxiv_id"])
WORKER_NAME = WORKER or ("local" if RUN_RANGES == "local" else "+".join(f"R{r}" for r in selected_ranges))
print(f"Worker: {WORKER_NAME} | {len(jobs):,} ID | {sum(len(j['version_dirs']) for j in jobs):,} phiên bản dự kiến")
preview = [{"arxiv_id": j["arxiv_id"], "range": j["range"], "số phiên bản": len(j["version_dirs"]),
            "thư mục": str(j.get("paper_dir") or "chưa có"), "lỗi phát hiện": "; ".join(j.get("discovery_errors", []))}
           for j in jobs[:20]]
display(pd.DataFrame(preview))

## 3. Kiểm thử nhỏ trước khi chạy corpus

Fixture offline gồm hai phiên bản có cùng câu nhưng khác heading/khóa citation và một phiên bản không có TeX. Kiểm tra bibliography giữ cả năm lẫn DOI và câu dùng chung vẫn gắn đúng cha. Phần này giả lập kết quả mô hình tách câu để kiểm tra tích hợp; không đánh giá độ chính xác SaT.

In [ ]:
if RUN_SMOKE_TEST:
    from tempfile import TemporaryDirectory
    from unittest.mock import patch
    with TemporaryDirectory() as temporary:
        base = Path(temporary)
        paper = base / "2301-00001"
        paper.mkdir()
        (paper / "metadata.json").write_text(json.dumps({"arxiv_id": "2301.00001", "num_versions": 3}), encoding="utf-8")
        for version, title, key, fields in [(1, "Introduction", "old", "year={2017}"),
                                           (2, "Background", "new", "doi={10.1000/attention}")]:
            folder = paper / f"v{version}"
            folder.mkdir()
            (folder / "main.tex").write_text(r"\documentclass{article}\begin{document}" +
                rf"\section{{{title}}}We study attention \cite{{{key}}}.\bibliography{{refs}}" +
                r"\end{document}", encoding="utf-8")
            (folder / "refs.bib").write_text(f"@article{{{key},title={{Attention Is All You Need}},"
                f"author={{Vaswani, Ashish}},{fields}}}", encoding="utf-8")
        (paper / "v3").mkdir()
        with patch("src.hierarchy._sentence_model") as model:
            model.return_value.split.side_effect = lambda text, **kwargs: [text]
            demo = parse_publication(discover_publications(base)[0], enrich=False)
        assert len(demo["bibtex"]) == 1 and demo["bibtex"][0]["year"] == 2017
        assert demo["bibtex"][0]["doi"] == "10.1000/attention"
        sentence = [r for r in demo["elements"] if r["kind"] == "sentence"]
        assert len(sentence) == 1
        edges = [r for r in demo["hierarchy"] if r["element_id"] == sentence[0]["element_id"]]
        assert len(edges) == 2 and edges[0]["parent_id"] != edges[1]["parent_id"]
        assert any(r["version"] == 3 and r["status"] == "empty" for r in demo["version_status"])
        print("Kiểm thử tích hợp đã đạt: hợp nhất reference, dùng chung câu, đúng cha theo phiên bản, giữ phiên bản rỗng.")

## 4. Chạy pipeline và lưu checkpoint

Các bài có thể chạy song song bằng nhiều tiến trình; mọi phiên bản của cùng bài vẫn được xử lý và hợp nhất trong cùng worker, giữ nguyên phương pháp parser. Worker tái sử dụng mô hình SaT CPU, còn tiến trình chính ghi Parquet/checkpoint và thống kê. Số bài đang chạy được giới hạn, không đẩy cả corpus vào hàng đợi. Bảng tổng được ghép theo batch và giữ thứ tự bài ổn định dù công việc hoàn thành khác thứ tự.

Khi `ADAPTIVE_RAM=True`, chương trình chừa RAM hệ thống, đo RSS của worker và cập nhật mức dự trù. Khi RAM giảm, chương trình ngừng cấp thêm bài và thu hồi tiến trình đang rảnh; bài đang chạy được hoàn tất. Nếu không còn bài đang chạy mà RAM vẫn không đủ sau `RAM_WAIT_TIMEOUT_SEC`, phiên lưu phần đã xong và báo lỗi để có thể chạy tiếp sau. Đây là cơ chế giảm nguy cơ hết RAM, không phải hạn mức RAM cứng: một bài quá lớn vẫn có thể vượt dự trù. `NUM_WORKERS=1` chạy trực tiếp tuần tự, không dùng bộ điều phối RAM.

Ở chế độ song song, kết quả được ghi bởi một luồng nền để việc ghi Parquet/checkpoint không chặn việc giao bài cho worker; số kết quả chờ ghi được giới hạn để RAM ổn định. `PAPER_TIMEOUT_SEC` đặt thời gian tối đa cho một bài, tính từ lúc giao bài (gồm cả khởi động worker mới và lần nạp mô hình đầu tiên). Bài vượt thời hạn bị dừng, ghi `error` kèm lý do và được thử lại ở phiên sau; worker thay thế được tạo khi cần. Giới hạn này chỉ áp dụng khi chạy song song, vì `NUM_WORKERS=1` chạy trong chính tiến trình notebook.

Trong mỗi worker, kết quả tách câu được lưu theo nội dung đoạn văn, nên đoạn lặp lại giữa các phiên bản của cùng bài chỉ chạy SaT một lần; kết quả giống với khi chạy lại mô hình. Khi khử trùng element, nội dung được chuẩn hóa khoảng trắng trước khi so khớp (mọi chuỗi khoảng trắng thành một dấu cách, trừ môi trường nguyên văn như `verbatim` và `\verb`), đúng yêu cầu so khớp sau cleanup bỏ qua khác biệt định dạng/khoảng trắng; bảng `elements` lưu nội dung đã chuẩn hóa.

Cảnh báo lỗi thời của thư viện (`DeprecationWarning`, `FutureWarning`, ...) không phản ánh dữ liệu nên chỉ được đếm trong `dependency_warning_count` của từng bài, không ghi thành dòng trong `warnings.csv`. Một nội dung lặp lại dưới nhiều cha trong cùng phiên bản (ví dụ đoạn "Then" đứng trước nhiều công thức) được gộp thành một cảnh báo mỗi phiên bản và không bật `review_required`, vì mọi cạnh đều được giữ. Đường dẫn trong thông điệp được ghi tương đối so với thư mục dữ liệu.

**Kiểm tra checkpoint.** Trước khi parse, mỗi bài được băm nội dung nguồn (`.tex`, `.bib`, `metadata.json`) cùng mã nguồn parser và tùy chọn; chữ ký trùng và trạng thái `ok`/`empty` thì kết quả cũ được dùng lại. Sáu tệp output của bài được kiểm tra theo `VERIFY_CHECKPOINTS`: `"size"` so kích thước với lúc ghi (một lần `stat` mỗi tệp), phát hiện tệp thiếu hoặc ghi dở; `"full"` băm lại SHA-256 và đếm dòng Parquet, phát hiện cả tệp bị sửa mà kích thước không đổi nhưng phải đọc hết từng tệp, chậm trên Google Drive. Tệp output được ghi kiểu atomic và trước tệp state, nên `"size"` đủ cho các lần chạy tiếp; `"full"` dành cho lần kiểm tra cuối. Toàn bộ bản ghi checkpoint của worker được chụp vào `.state/<WORKER>/_manifest.json` khi phiên kết thúc, để phiên sau đọc một tệp thay vì một tệp mỗi bài. Bản chụp bị xóa ngay khi phiên bắt đầu, nên một phiên bị ngắt đột ngột (mất runtime) không để lại bản chụp cũ: phiên sau đọc từng tệp state như cũ. Thời gian kiểm tra checkpoint, số bài dùng lại từ bản chụp/tệp state và chế độ kiểm tra nằm trong `summary_stats.json` (`run.checkpoint`); thời gian giải nén dữ liệu, nếu dùng bản tar, nằm ở `run.data_preparation` và nên được cộng vào thời gian end-to-end khi báo cáo.

Hết `MAX_RUN_HOURS` thì không cấp bài mới, nhưng chờ các bài đã cấp hoàn tất. Interrupt dọn các tiến trình do phiên tạo; bài chưa ghi checkpoint được đánh dấu pending. Lịch sử tăng/giảm worker và chờ RAM nằm trong `events.jsonl` và thông tin song song của phiên.

`paper_success` đúng nếu mọi phiên bản dự kiến được xử lý đầy đủ hoặc xác nhận rỗng. `ok` có thể đi kèm ghi chú (`notice`) về metadata đã hợp nhất hoặc quan hệ lặp đã được giữ nguyên; ghi chú này không làm bài bị parse lại. `review_required=True` cho biết chất lượng metadata/schema còn cần xem xét, tách biệt với việc pipeline đã hoàn tất.

`partial` dành cho kết quả còn thiếu hoặc chưa chắc đầy đủ: file nội dung chính không có, macro/conditional chưa giải được, LaTeX lỗi hoặc khóa citation mơ hồ. `error` là bước xử lý thất bại. Các trường hợp này tiếp tục được thử lại và không được tính thành công đầy đủ. Xem `issues.csv` để tìm việc cần khắc phục; `warnings.csv` giữ mọi chẩn đoán kèm `affects_status` và `requires_review`. Không thể tái tạo chính xác tài liệu tham khảo từ một `.bbl` đã bị thiếu bằng cách đổi nhãn trạng thái.

In [ ]:
def show_progress(done, total, stats):
    if done == 1 or done % 25 == 0 or done == total:
        print(f"[{done:,}/{total:,}] {stats['arxiv_id']} — {stats['status']} | "
              f"element: {stats.get('elements_before', 0)} → {stats.get('elements_after', 0)} | "
              f"notices: {stats.get('notice_count', 0)}, warnings affecting completeness: {stats.get('blocking_warning_count', 0)}")
        if stats['status'] not in {'ok', 'empty'}:
            print("  Reason:", stats.get('message', '')[:700])

if RUN_PIPELINE:
    result = run_parser(jobs, OUTPUT_DIR, worker=WORKER_NAME, ranges=RANGES,
        enrich=ENABLE_ENRICHMENT, fuzzy_threshold=FUZZY_THRESHOLD, resume=RESUME,
        max_run_hours=MAX_RUN_HOURS, monitor_interval=MONITOR_EVERY_SEC,
        disk_every=DISK_EVERY_PAPERS, progress=show_progress,
        num_workers=NUM_WORKERS, max_workers=MAX_WORKERS, adaptive_ram=ADAPTIVE_RAM,
        reserve_ram_gb=RESERVE_RAM_GB, ram_per_worker_gb=RAM_PER_WORKER_GB,
        ram_wait_timeout_sec=RAM_WAIT_TIMEOUT_SEC, paper_timeout_sec=PAPER_TIMEOUT_SEC,
        verify_checkpoints=VERIFY_CHECKPOINTS, data_preparation=DATA_PREPARATION)
    summary = result["summary"]
else:
    summary_path = OUTPUT_DIR / "report_exports" / WORKER_NAME / "summary_stats.json"
    if not summary_path.exists():
        raise FileNotFoundError("No reports available. Set RUN_PIPELINE=True for the initial run.")
    summary = json.loads(summary_path.read_text(encoding="utf-8"))

REPORTS_DIR = OUTPUT_DIR / "report_exports" / WORKER_NAME
TABLES_DIR = OUTPUT_DIR / "tables" / WORKER_NAME
display(pd.DataFrame([{
    "IDs in scope": summary["target_ids"], "successful papers": summary["papers_success"],
    "success rate": summary["success_rate"], "elements after deduplication": summary["elements_after"],
    "references after deduplication": summary["bibtex_after"], "stop reason": summary["run"]["stop_reason"],
    "run duration (seconds)": summary["run"]["duration_sec"],
    "checkpoint check (seconds)": summary["run"].get("checkpoint", {}).get("sec"),
    "data extraction this session (seconds)": summary["run"].get("data_preparation", {}).get("extract_sec_this_session"),
    "peak RSS (MiB)": summary["run"]["peak_rss_bytes"] / 2**20,
    "Parquet tables (MiB)": summary["table_bytes"] / 2**20,
}]))
if summary["run"].get("checkpoint"):
    print("Checkpoint:", summary["run"]["checkpoint"])
if summary["run"].get("parallel"):
    print("CPU/RAM scheduling:", summary["run"]["parallel"])
if summary["run"]["stop_reason"] == "error":
    print("Run error:", summary["run"].get("error"), "— check parser.log before using the tables.")

## 4b. Hậu xử lý `bibtex` bằng AnyStyle

Bước tùy chọn chạy **sau** khi parse xong, trên toàn bộ bài của worker, theo lô; không sửa partition hay checkpoint của parser. AnyStyle (bộ tách trích dẫn dùng mô hình CRF) chỉ nhận các `\bibitem` có dấu hiệu tách sai: thiếu tiêu đề hoặc tác giả, tiêu đề trông như tên tạp chí/hội nghị, tên tác giả dài bất thường. Tiêu đề AnyStyle trả về phải xuất hiện trong văn bản `\bibitem` gốc mới được nhận. Mọi mục được gửi trong một lần gọi CLI theo khối 2.000 dòng, vì chi phí chính là khởi động Ruby.

Đầu vào duy nhất là văn bản nguồn của bài báo. Parser và bước này **không** dùng danh sách tham khảo đã thu thập (`references.json`) hay tra cứu dịch vụ ngoài: Milestone 3 ghép chính các entry `bibtex` với danh sách đó, nên mọi trường suy ra từ nó sẽ làm lộ nhãn.

Kết quả được cache trong `REFINE_CACHE_DIR/anystyle.jsonl`; lần chạy sau chỉ gọi CLI cho phần chưa có. `tables/<WORKER>/bibtex.parquet` được thay bằng bảng đã tinh chỉnh (đánh dấu trong metadata Parquet), bảng do parser tạo giữ ở `bibtex_parsed.parquet`, từng thay đổi (trường, giá trị cũ/mới, lý do) nằm trong `bibtex_refinement.jsonl`. Chạy lại `run_parser` sẽ ghi lại `bibtex.parquet` từ partition, nên cần chạy lại cả mục này. Bước này không khử trùng lại. Thời gian của mục này được báo riêng, không tính vào thời gian parse.


In [ ]:
if REFINE_BIBTEX:
    if REFINE_WITH_ANYSTYLE and shutil.which("anystyle") is None:
        if IN_COLAB:
            tick = time.perf_counter()
            installed = subprocess.run("apt-get -qq update && apt-get -qq install -y ruby-dev build-essential "
                                       "> /dev/null && gem install --no-document anystyle-cli > /dev/null",
                                       shell=True, capture_output=True, text=True)
            # Cài thất bại không dừng Run all: bước AnyStyle tự bỏ qua và ghi lý do vào thống kê.
            print(f"Cài anystyle-cli: {'xong' if installed.returncode == 0 else 'thất bại'} "
                  f"sau {time.perf_counter() - tick:.0f} s", installed.stderr.strip()[-300:])
        else:
            print("Không có lệnh anystyle (cần Ruby và `gem install anystyle-cli`); bỏ qua bước AnyStyle.")
    from src.bibtex_refine import refine_bibliography
    tick = time.perf_counter()
    refinement = refine_bibliography(
        OUTPUT_DIR, worker=WORKER_NAME,
        cache_dir=REFINE_CACHE_DIR or OUTPUT_DIR / "refine_cache",
        use_anystyle=REFINE_WITH_ANYSTYLE, progress=print)
    print(f"Hậu xử lý bibtex: {time.perf_counter() - tick:.1f} s")
    if refinement["anystyle"].get("skipped"):
        print("AnyStyle:", refinement["anystyle"]["skipped"])
    display(pd.DataFrame([{k: v for k, v in refinement["anystyle"].items() if k != "skipped"}]))
    display(pd.Series(refinement["changes_by_field"], name="số thay đổi", dtype="int64"))
    print(f"{refinement['rows_changed']:,}/{refinement['rows']:,} entry được tinh chỉnh; "
          f"chi tiết: {OUTPUT_DIR / 'tables' / WORKER_NAME / 'bibtex_refinement.jsonl'}")


## 5. Thống kê độ phủ, khử trùng và hiệu năng

Các CSV/JSON tương ứng hướng dẫn `statistics_data_output.md`, được điều chỉnh cho Milestone 2. `per_version.csv` bổ sung độ phủ phiên bản. `success_rate` dùng toàn bộ ID của phạm vi làm mẫu số; bản ghi `pending` do hết giờ hoặc Interrupt vẫn được giữ. Thời gian parse từ checkpoint khác chi phí của lần chạy hiện tại. Khi chạy song song, tổng thời gian xử lý từng bài có thể lớn hơn thời gian thực của phiên vì nhiều bài chạy cùng lúc; dùng duration_sec của phiên khi so sánh tốc độ.

RSS là tổng bộ nhớ cư trú của tiến trình Python và tiến trình con, không phải RAM của toàn máy. Đo theo chu kỳ có thể bỏ qua đỉnh rất ngắn; thời gian lần chạy đầu bao gồm tải mô hình trong bước parse. `output_snapshot` đo thư mục output tại thời điểm ghi nhận, gồm phân mảnh, bảng tổng và log. `table_bytes` chỉ đo ba bảng tổng để báo dung lượng dataset, tránh cộng đôi phân mảnh.

In [ ]:
per_paper = pd.read_csv(REPORTS_DIR / "per_paper.csv", dtype={"arxiv_id": str})
per_version = pd.read_csv(REPORTS_DIR / "per_version.csv", dtype={"arxiv_id": str})
range_stats = pd.read_csv(REPORTS_DIR / "range_stats.csv")
display(range_stats)
display(per_paper.head(20))
display(per_version.head(20))
issues = per_paper[~per_paper["status"].isin(["ok", "empty"])]
print(f"{len(issues):,} papers have missing input, errors, or pending processing.")
display(issues.head(20))
diagnostics = pd.read_csv(REPORTS_DIR / "warnings.csv", dtype={"arxiv_id": str})
actionable_versions = pd.read_csv(REPORTS_DIR / "issues.csv", dtype={"arxiv_id": str})
print("Versions requiring attention:")
display(actionable_versions.head(30))
print("Quality notices and warnings (only affects_status=True indicates incomplete results):")
display(diagnostics.head(30))

In [ ]:
import matplotlib.pyplot as plt

FIGURES_DIR = REPORTS_DIR / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
per_paper["status"].value_counts().plot.bar(ax=axes[0], title="Paper status") if len(per_paper) else axes[0].set_title("No papers")
axes[0].set_ylabel("Number of papers")
if len(per_version):
    per_version["status"].value_counts().plot.bar(ax=axes[1], title="Version status")
else:
    axes[1].set_title("No versions")
axes[1].set_ylabel("Number of versions")
pd.DataFrame({"Before": [summary["elements_before"], summary["bibtex_before"]],
              "After": [summary["elements_after"], summary["bibtex_after"]]},
             index=["Element", "Reference"]).plot.bar(ax=axes[2], title="Before and after deduplication")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "coverage_dedup.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
current = per_paper[per_paper["reused"].eq(False)]
times = pd.to_numeric(current["elapsed_sec"], errors="coerce").dropna()
if len(times):
    axes[0].hist(times, bins=min(30, max(1, len(times))))
axes[0].set(title="Parsing time for papers without reused checkpoints", xlabel="Seconds", ylabel="Number of papers")
monitor_path = REPORTS_DIR / "monitor.csv"
if monitor_path.exists():
    monitor = pd.read_csv(monitor_path)
    for run_id, group in monitor.groupby("run_id"):
        axes[1].plot(pd.to_datetime(group["minute"], utc=True), group["max_rss_bytes"] / 2**20,
                     label=run_id)
    if len(monitor):
        axes[1].legend(fontsize=7)
axes[1].set(title="Peak RSS per minute", xlabel="UTC time", ylabel="MiB")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "performance.png", dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)

## 6. Bảng đầu ra và cách đọc

```text
statistics_data/parser/
├── tables/<WORKER>/
│   ├── elements.parquet       # element_id, arxiv_id, kind, content
│   ├── hierarchy.parquet      # arxiv_id, version, element_id, parent_id
│   ├── bibtex.parquet         # chín cột theo parsing.md (đã tinh chỉnh nếu chạy mục 4b)
│   ├── bibtex_parsed.parquet  # bảng do parser tạo, trước mục 4b
│   ├── bibtex_refinement.jsonl # từng thay đổi của mục 4b
│   └── papers/<ID>/           # phân mảnh, key_map.json, bibtex_debug.jsonl, warnings.jsonl
├── report_exports/<WORKER>/   # per_paper.csv, per_version.csv, summary_stats.json,
│                            # range_stats.csv, ranges.csv, category_counts.csv,
│                            # runs.csv, monitor.csv, disk.csv, warnings.csv, issues.csv, README.md, figures/,
│                            # bibtex_refinement_summary.json
├── refine_cache/              # anystyle.jsonl (mục 4b)
├── run_logs/<WORKER>/         # runs.jsonl, timings.jsonl, events.jsonl, parser.log,
│                            # monitor_<run_id>.csv, disk_<run_id>.csv
└── .state/<WORKER>/           # checkpoint có chữ ký nguồn/cấu hình
```

`kind` và `entry_type` có kiểu dictionary, `version`/`year` là integer, `authors` là danh sách chuỗi. Bảng rỗng vẫn giữ schema. Root có dòng trong `elements` nhưng không sinh cạnh cha. Không đọc toàn bộ `papers/` cùng với bảng tổng bằng một glob Parquet vì sẽ đếm hai lần.

**Nội dung `elements`.** Nút gốc `document` của mỗi phiên bản mang tiêu đề bài (lấy từ `\title`, bỏ `\thanks` và ngắt dòng); nếu không có tiêu đề thì nội dung rỗng. Thẻ `\begin`/`\end` của các môi trường không thuộc nhóm công thức, hình, danh sách hay nguyên văn (ví dụ môi trường định lý tự định nghĩa) là ranh giới khối và không xuất hiện trong nội dung câu. Văn bản trước và sau một công thức display là các câu riêng, vì công thức là một lá độc lập của cây.

**Nội dung `bibtex`.** `title`, `journal` và `authors` được chuẩn hóa từ LaTeX sang Unicode (dấu như `{\"u}` thành `ü`, bỏ ngoặc bảo vệ chữ hoa và lệnh định dạng); công thức `$...$` được giữ nguyên. Tên tác giả theo thứ tự `Tên Họ`; tên tổ chức đặt trong ngoặc ở nguồn BibTeX giữ nguyên. arXiv ID được lấy từ `eprint`, DOI `10.48550/arXiv.*`, URL arxiv.org hoặc chuỗi `arXiv:...` trong các trường văn bản, và dùng làm khóa nhận diện khi hợp nhất. Năm đọc từ `year` hoặc `date`; nếu không phiên bản nào có năm nhưng có arXiv ID, năm được suy từ ID (năm nộp bản đầu) và ghi `inferred_fields=["year"]` trong `bibtex_debug.jsonl`. Bảng giữ mọi entry của bibliography; cờ `cited` trong `bibtex_debug.jsonl` và `bibtex_cited` trong thống kê cho biết entry nào được `\cite` trong nội dung đã parse (`\nocite{*}` tính là tất cả).

**Tách `\bibitem`.** Mục có `\newblock` được tách theo khối của BibTeX: khối đầu là tác giả, khối thứ hai là tiêu đề, phần còn lại là nơi đăng. Mục không có `\newblock` dùng dấu ngoặc kép, chữ in nghiêng và dấu câu (cả dạng `Tác giả. Tiêu đề. In \emph{Hội nghị}`); style tác giả–năm (`Tác giả. 2024. Tiêu đề.`) được nhận ra để năm không lẫn vào tên tác giả cuối. Chỉ văn bản nguồn được dùng; danh sách tham khảo đã thu thập không tham gia vì đó là dữ liệu Milestone 3 ghép với. `bibtex_debug.jsonl` giữ văn bản `\bibitem` (`raw_text`) cho bước 4b. Thống kê từng bài có `cite_keys` (số khóa được `\cite`) và `cite_keys_unresolved` (khóa không có entry nào, thường vì bibliography chỉ có ở dạng `.bbl` mà Milestone 1 không giữ); `summary_stats.json` cộng dồn trong `bibliography`.

**Element dùng chung giữa các phiên bản.** Lá (câu, công thức, hình/bảng) trùng nội dung sau cleanup dùng một `element_id`. Nút có con (gốc, heading, list, item) được so thêm theo thứ tự xuất hiện trong phiên bản: heading "Proof" thứ k của v1 và của v2 dùng chung ID, nhưng hai heading "Proof" hay hai list của cùng một phiên bản là hai nút riêng, nên con của chúng không bị gộp.

Log thời gian ghi riêng từng bước; cảnh báo/lỗi nằm trong trạng thái và log. Lịch sử phiên của worker được giữ qua các lần chạy, còn ba bảng tổng và báo cáo phạm vi được cập nhật cho **lần xuất hiện tại**. Không coi cài đặt, clone repo và mount Drive là thời gian parse corpus.

In [ ]:
# Chỉ lấy batch đầu để xem mẫu, tránh nạp cả corpus vào RAM.
for name in ("elements", "hierarchy", "bibtex"):
    path = TABLES_DIR / f"{name}.parquet"
    with pq.ParquetFile(path) as parquet:
        print(f"\n{name}: {parquet.metadata.num_rows:,} dòng | {path}")
        print(parquet.schema_arrow)
        first_batch = next(parquet.iter_batches(batch_size=5), None)
        if first_batch is not None:
            display(first_batch.to_pandas())
print("Report exports:", REPORTS_DIR)